In [ ]:
# Import necessary libraries
import json
import anthropic


In [ ]:
# Import environment variables from .env file
from dotenv import load_dotenv 
load_dotenv()

In [ ]:

def add_user_messages(messages, message):
    messages.append({"role": "user", "content": message})

def add_assistant_messages(messages, message):
    messages.append({"role": "assistant", "content": message})
    
def chat(messages, model="claude-haiku-4-5", system=None, temperature=1.0, stop_sequences=[]):
    parameters = {
        "model": model,
        "max_tokens": 512,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }
    
    if system:
        parameters["system"] = system
    
    client = anthropic.Anthropic()
    response = client.messages.create(**parameters)
    return response.content[0].text

In [ ]:
def run_prompt(prompt_inputs):
    prompt = f"""
        What should this person eat?

        - Height: {prompt_inputs["height"]}
        - Weight: {prompt_inputs["weight"]}
        - Goal: {prompt_inputs["goal"]}
        - Dietary restrictions: {prompt_inputs["restrictions"]}
        """
    messages = []
    add_user_messages(messages, prompt)
    return chat(messages)

def grade_by_model(solution_criteria, output, specs):
    prompt = f"""
        Overview output and assess content to meeting on solution criteria based on specs.
        Output:
        <output>
        {output}
        </output>
        Criteria:
        <criteria>
        {solution_criteria}
        </criteria>
        Specs:
        <specs>
        {specs}
        </spec>

        Provide your evaluation as JSON object:
        - grade: integer from 0 to 10, where 10 is the best grade.
        - comment: short explanation of the grade.

        # Restrictions:
        - Output should be without any markdown or code block, only JSON object. 
        - Don't wrap output in any markdown. Don't use any backticks.
        """
    messages = []
    add_user_messages(messages, prompt)
    grade = chat(messages, system="""
                 Output should be without any markdown or code block, only JSON object. 
                 Don't wrap output in any markdown. Don't use any backticks.
                 """)
    print(grade)
    return json.loads(grade)

class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=1):
        self.max_concurrent_tasks = max_concurrent_tasks

    def generate_dataset(
        self,
        task,
        prompt_specs={},
        output_file="dataset-prompt-techniques.json",
        num_cases=3
    ):
        messages = []
        add_user_messages(
            messages, 
            f"""
            Generate a dataset of unique {num_cases} elements for task:
            <task> 
            {task}
            </task> 
            
            Output has to include the below specification:
            <specification>
            {prompt_specs}
            </specification>

            Include into output a solution criteria field named `solution_criteria` for each case. 
            It should be a key point that defines whether the solution is correct or not. 

            # Restrictions: 
            - The output should be a JSON an array of strings.
            - Don't wrap it in any markdown or code block.
            - Don't use any backticks in output.
            """
            )
        response = chat(messages, system="Output should be a JSON array of strings. Don't wrap it in any markdown or code block. Don't use any backticks in output.")
        print(response)
        with open(output_file, "w") as f:
            json.dump(json.loads(response), f, indent=4)
    
    def run_evaluation(self, dataset):
        with open(dataset, "r") as f:
            tasks = json.load(f)

        results = []
        for task in tasks: 
            output = run_prompt(task)
            results.append({
                "task": task,
                "output": output,
                "grade_by_model": grade_by_model(task["solution_criteria"], output, tasks),
            })
            
        print(results)
        with open("evaluation_results.json", "w") as f:
            json.dump(results, f, indent=4)


In [ ]:
evaluator = PromptEvaluator(max_concurrent_tasks=5)
evaluator.generate_dataset(
    task="Write compact, consice 1 day meal plan for a single athlete",
    prompt_specs="""
    {
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete"
    }
    """,
)

evaluator.run_evaluation("dataset-prompt-techniques.json")